In [50]:
# ============================================================================
# Cell 1: Data Initialization (Complete with All Functions)
# ============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import re

class QuantizationDataManager:
    """
    Manages quantization experimental data for multiple activation functions
    Focus on RMSE as the primary error metric
    Automatically parses configuration strings to extract bit-widths
    """
    
    def __init__(self):
        self.data = pd.DataFrame()
    
    @staticmethod
    def _parse_bits(config_str):
        """
        Parse configuration string to extract bit-widths
        
        Examples:
        - 'FP64_FP64' -> (64, 64)
        - 'FP32_FP16' -> (32, 16)
        - 'Fixed2_14_Fixed2_14' -> (16, 16)  # 2+14=16, 2+14=16
        - 'Fixed2_14_Fixed4_12' -> (16, 16)  # 2+14=16, 4+12=16
        - 'Fixed2_30_Fixed2_14' -> (32, 16)  # 2+30=32, 2+14=16
        
        Returns:
        --------
        tuple: (endpoint_bits, param_bits)
        """
        parts = config_str.split('_')
        
        # Floating-point: FP{bits}_FP{bits}
        if config_str.startswith('FP'):
            endpoint_bits = int(parts[0][2:])  # FP64 -> 64
            param_bits = int(parts[1][2:])      # FP32 -> 32
            return endpoint_bits, param_bits
        
        # Fixed-point: Fixed{int}_{frac}_Fixed{int}_{frac}
        elif config_str.startswith('Fixed'):
            match = re.findall(r'Fixed(\d+)_(\d+)', config_str)
            if len(match) == 2:
                # Endpoint: int1 + frac1
                endpoint_bits = int(match[0][0]) + int(match[0][1])
                # Parameter: int2 + frac2
                param_bits = int(match[1][0]) + int(match[1][1])
                return endpoint_bits, param_bits
        
        return None, None
    
    @staticmethod
    def _classify_config(config_str, endpoint_bits, param_bits):
        """
        Classify configuration type
        
        Returns:
        --------
        str: 'Float', 'Fixed', 'Mixed', or None (for hybrids)
        """
        has_fp = 'FP' in config_str
        has_fixed = 'Fixed' in config_str
        
        # Hybrid (FP + Fixed) -> exclude
        if has_fp and has_fixed:
            return None
        
        # Pure Float
        if has_fp:
            return 'Float' if endpoint_bits == param_bits else 'Mixed'
        
        # Pure Fixed
        if has_fixed:
            return 'Fixed' if endpoint_bits == param_bits else 'Mixed'
        
        return None
    
    @staticmethod
    def _get_display_name(config_str, endpoint_bits, param_bits, config_type):
        """
        Generate display name for configuration
        
        Examples:
        - FP64_FP64 -> 'FP64'
        - FP32_FP16 -> 'Mixed FP32/16'
        - Fixed2_14_Fixed2_14 -> 'Fixed-16bit'
        - Fixed2_14_Fixed4_12 -> 'Fixed-16bit'
        - Fixed2_30_Fixed2_14 -> 'Mixed Fixed-32/16'
        """
        if config_type is None:
            return config_str
        
        has_fp = 'FP' in config_str
        
        if config_type == 'Float':
            return f'FP{endpoint_bits}'
        elif config_type == 'Fixed':
            return f'Fixed-{endpoint_bits}bit'
        elif config_type == 'Mixed':
            prefix = 'FP' if has_fp else 'Fixed-'
            return f'Mixed {prefix}{endpoint_bits}/{param_bits}'
        
        return config_str
    
    def add_function_data(self, function_name, experimental_results):
        """
        Add experimental results for an activation function
        
        Parameters:
        -----------
        function_name : str
            Name of activation function (e.g., 'Tanh', 'Exp', 'GELU', 'SiLU')
        experimental_results : list of dict
            Each dict contains:
            - config: str (e.g., 'FP32_FP32', 'Fixed2_14_Fixed4_12')
            - rmse: float (required)
            - avg_mae: float (optional)
            - max_mae: float (optional)
            - compression: float (optional)
        """
        new_data = []
        
        for result in experimental_results:
            config = result['config']
            
            # Parse bit-widths
            endpoint_bits, param_bits = self._parse_bits(config)
            if endpoint_bits is None:
                print(f"Warning: Cannot parse config '{config}' - skipping")
                continue
            
            # Classify configuration
            config_type = self._classify_config(config, endpoint_bits, param_bits)
            
            # Generate display name
            display_name = self._get_display_name(config, endpoint_bits, param_bits, config_type)
            
            # Calculate total bits
            total_bits = endpoint_bits + param_bits
            
            # Calculate MAE/RMSE ratio if MAE available
            avg_mae = result.get('avg_mae')
            rmse = result['rmse']
            mae_rmse_ratio = avg_mae / rmse if avg_mae and rmse > 0 else None
            
            new_data.append({
                'function': function_name,
                'config': config,
                'config_display': display_name,
                'config_type': config_type,
                'endpoint_bits': endpoint_bits,
                'param_bits': param_bits,
                'total_bits': total_bits,
                'rmse': rmse,
                'avg_mae': avg_mae,
                'max_mae': result.get('max_mae'),
                'mae_rmse_ratio': mae_rmse_ratio,
                'compression': result.get('compression', 1.0),
                'is_hybrid': config_type is None
            })
        
        new_df = pd.DataFrame(new_data)
        self.data = pd.concat([self.data, new_df], ignore_index=True)
        
        print(f"✓ Added {len(new_data)} configurations for '{function_name}'")
        return self
    
    def get_data(self, functions=None, config_types=None, exclude_hybrids=True):
        """
        Retrieve filtered data
        
        Parameters:
        -----------
        functions : list of str, optional
            Filter by function names
        config_types : list of str, optional
            Filter by types ('Float', 'Fixed', 'Mixed')
        exclude_hybrids : bool
            Exclude Float-Fixed hybrid configurations
        
        Returns:
        --------
        DataFrame
        """
        df = self.data.copy()
        
        if exclude_hybrids:
            df = df[~df['is_hybrid']]
        
        if functions is not None:
            df = df[df['function'].isin(functions)]
        
        if config_types is not None:
            df = df[df['config_type'].isin(config_types)]
        
        return df.sort_values(['total_bits', 'endpoint_bits'])
    
    def get_summary(self):
        """Print summary statistics"""
        if self.data.empty:
            print("No data loaded")
            return
        
        print("\n" + "="*80)
        print("DATA SUMMARY")
        print("="*80)
        print(f"Total configurations: {len(self.data)}")
        print(f"\nFunctions ({len(self.data['function'].unique())}):")
        for func in sorted(self.data['function'].unique()):
            count = len(self.data[self.data['function'] == func])
            print(f"  • {func}: {count} configs")
        
        print(f"\nConfiguration Types:")
        df_clean = self.data[~self.data['is_hybrid']]
        for ctype in sorted(df_clean['config_type'].unique()):
            count = len(df_clean[df_clean['config_type'] == ctype])
            print(f"  • {ctype}: {count} configs")
        
        if self.data['is_hybrid'].any():
            hybrid_count = self.data['is_hybrid'].sum()
            print(f"  • Hybrid (excluded): {hybrid_count} configs")
        
        print(f"\nBit-width Range:")
        print(f"  • Min: {df_clean['total_bits'].min()} bits")
        print(f"  • Max: {df_clean['total_bits'].max()} bits")
        
        print(f"\nRMSE Range:")
        print(f"  • Min: {df_clean['rmse'].min():.2e}")
        print(f"  • Max: {df_clean['rmse'].max():.2e}")
        print("="*80 + "\n")
    
    def save_data(self, filepath='quantization_data.csv'):
        """Save data to CSV"""
        self.data.to_csv(filepath, index=False)
        print(f"✓ Data saved to {filepath}")
    
    def load_data(self, filepath='quantization_data.csv'):
        """Load data from CSV"""
        if Path(filepath).exists():
            self.data = pd.read_csv(filepath)
            print(f"✓ Data loaded from {filepath}")
        else:
            print(f"✗ File not found: {filepath}")


# ============================================================================
# Initialize and Load All Data
# ============================================================================

# Create data manager
data_manager = QuantizationDataManager()

# Tanh function data
tanh_results = [
    # Floating-Point
    {'config': 'FP64_FP64', 'avg_mae': 6.798826e-05, 'max_mae': 1.448115e-04, 'rmse': 7.548510e-05, 'compression': 1.00},
    {'config': 'FP32_FP32', 'avg_mae': 6.80e-05, 'max_mae': 1.45e-04, 'rmse': 7.55e-05, 'compression': 2.00},
    {'config': 'FP16_FP16', 'avg_mae': 1.05e-04, 'max_mae': 3.19e-04, 'rmse': 1.26e-04, 'compression': 4.00},
    {'config': 'FP64_FP32', 'avg_mae': 6.80e-05, 'max_mae': 1.45e-04, 'rmse': 7.55e-05, 'compression': 1.43},
    {'config': 'FP64_FP16', 'avg_mae': 1.05e-04, 'max_mae': 3.19e-04, 'rmse': 1.26e-04, 'compression': 1.82},
    {'config': 'FP32_FP16', 'avg_mae': 1.05e-04, 'max_mae': 3.19e-04, 'rmse': 1.26e-04, 'compression': 2.86},
    
    # Fixed-Point
    {'config': 'Fixed2_14_Fixed2_14', 'avg_mae': 6.61e-05, 'max_mae': 1.47e-04, 'rmse': 7.53e-05, 'compression': 4.00},
    {'config': 'Fixed2_22_Fixed2_22', 'avg_mae': 6.80e-05, 'max_mae': 1.45e-04, 'rmse': 7.55e-05, 'compression': 2.67},
    {'config': 'Fixed2_30_Fixed2_30', 'avg_mae': 6.80e-05, 'max_mae': 1.45e-04, 'rmse': 7.55e-05, 'compression': 2.00},
    {'config': 'Fixed2_38_Fixed2_38', 'avg_mae': 6.80e-05, 'max_mae': 1.45e-04, 'rmse': 7.55e-05, 'compression': 1.60},
    {'config': 'Fixed2_46_Fixed2_46', 'avg_mae': 6.80e-05, 'max_mae': 1.45e-04, 'rmse': 7.55e-05, 'compression': 1.33},
    {'config': 'Fixed2_62_Fixed2_62', 'avg_mae': 6.80e-05, 'max_mae': 1.45e-04, 'rmse': 7.55e-05, 'compression': 1.00},
    
    # Mixed Fixed-Point
    {'config': 'Fixed2_62_Fixed2_30', 'avg_mae': 6.80e-05, 'max_mae': 1.45e-04, 'rmse': 7.55e-05, 'compression': 1.43},
    {'config': 'Fixed2_46_Fixed2_22', 'avg_mae': 6.80e-05, 'max_mae': 1.45e-04, 'rmse': 7.55e-05, 'compression': 1.90},
    {'config': 'Fixed2_30_Fixed2_14', 'avg_mae': 6.61e-05, 'max_mae': 1.47e-04, 'rmse': 7.53e-05, 'compression': 2.86},
]

# Exp function data
exp_results = [
    # Floating-Point
    {'config': 'FP64_FP64', 'avg_mae': 4.434179e-05, 'max_mae': 1.509635e-04, 'rmse': 5.178014e-05, 'compression': 1.00},
    {'config': 'FP32_FP32', 'avg_mae': 4.43e-05, 'max_mae': 1.51e-04, 'rmse': 5.18e-05, 'compression': 2.00},
    {'config': 'FP16_FP16', 'avg_mae': 2.13e-04, 'max_mae': 7.91e-04, 'rmse': 2.53e-04, 'compression': 4.00},
    {'config': 'FP64_FP32', 'avg_mae': 4.43e-05, 'max_mae': 1.51e-04, 'rmse': 5.18e-05, 'compression': 1.43},
    {'config': 'FP64_FP16', 'avg_mae': 2.13e-04, 'max_mae': 7.90e-04, 'rmse': 2.53e-04, 'compression': 1.82},
    {'config': 'FP32_FP16', 'avg_mae': 2.13e-04, 'max_mae': 7.90e-04, 'rmse': 2.53e-04, 'compression': 2.86},
    
    # Fixed-Point
    {'config': 'Fixed2_14_Fixed4_12', 'avg_mae': 7.37e-05, 'max_mae': 2.54e-04, 'rmse': 8.94e-05, 'compression': 4.00},
    {'config': 'Fixed2_22_Fixed4_20', 'avg_mae': 4.43e-05, 'max_mae': 1.51e-04, 'rmse': 5.17e-05, 'compression': 2.67},
    {'config': 'Fixed2_30_Fixed4_28', 'avg_mae': 4.43e-05, 'max_mae': 1.51e-04, 'rmse': 5.18e-05, 'compression': 2.00},
    {'config': 'Fixed2_38_Fixed4_36', 'avg_mae': 4.43e-05, 'max_mae': 1.51e-04, 'rmse': 5.18e-05, 'compression': 1.60},
    {'config': 'Fixed2_46_Fixed4_44', 'avg_mae': 4.43e-05, 'max_mae': 1.51e-04, 'rmse': 5.18e-05, 'compression': 1.33},
    {'config': 'Fixed2_62_Fixed4_60', 'avg_mae': 4.43e-05, 'max_mae': 1.51e-04, 'rmse': 5.18e-05, 'compression': 1.00},
    
    # Mixed Fixed-Point
    {'config': 'Fixed2_62_Fixed4_28', 'avg_mae': 4.43e-05, 'max_mae': 1.51e-04, 'rmse': 5.18e-05, 'compression': 1.43},
    {'config': 'Fixed2_46_Fixed4_20', 'avg_mae': 4.43e-05, 'max_mae': 1.51e-04, 'rmse': 5.17e-05, 'compression': 1.90},
    {'config': 'Fixed2_30_Fixed4_12', 'avg_mae': 7.36e-05, 'max_mae': 2.54e-04, 'rmse': 8.94e-05, 'compression': 2.86},
]

# GELU function data
gelu_results = [
    # Floating-Point
    {'config': 'FP64_FP64', 'avg_mae': 6.572215e-05, 'max_mae': 1.491397e-04, 'rmse': 7.375111e-05, 'compression': 1.00},
    {'config': 'FP32_FP32', 'avg_mae': 6.57e-05, 'max_mae': 1.49e-04, 'rmse': 7.37e-05, 'compression': 2.00},
    {'config': 'FP16_FP16', 'avg_mae': 1.22e-04, 'max_mae': 4.88e-04, 'rmse': 1.70e-04, 'compression': 4.00},
    {'config': 'FP64_FP32', 'avg_mae': 6.57e-05, 'max_mae': 1.49e-04, 'rmse': 7.37e-05, 'compression': 1.43},
    {'config': 'FP64_FP16', 'avg_mae': 1.22e-04, 'max_mae': 4.88e-04, 'rmse': 1.70e-04, 'compression': 1.82},
    {'config': 'FP32_FP16', 'avg_mae': 1.22e-04, 'max_mae': 4.88e-04, 'rmse': 1.70e-04, 'compression': 2.86},
    
    # Fixed-Point
    {'config': 'Fixed2_14_Fixed3_13', 'avg_mae': 7.11e-05, 'max_mae': 1.98e-04, 'rmse': 8.27e-05, 'compression': 4.00},
    {'config': 'Fixed2_22_Fixed3_21', 'avg_mae': 6.57e-05, 'max_mae': 1.49e-04, 'rmse': 7.37e-05, 'compression': 2.67},
    {'config': 'Fixed2_30_Fixed3_29', 'avg_mae': 6.57e-05, 'max_mae': 1.49e-04, 'rmse': 7.38e-05, 'compression': 2.00},
    {'config': 'Fixed2_38_Fixed3_37', 'avg_mae': 6.57e-05, 'max_mae': 1.49e-04, 'rmse': 7.38e-05, 'compression': 1.60},
    {'config': 'Fixed2_46_Fixed3_45', 'avg_mae': 6.57e-05, 'max_mae': 1.49e-04, 'rmse': 7.38e-05, 'compression': 1.33},
    {'config': 'Fixed2_62_Fixed3_61', 'avg_mae': 6.57e-05, 'max_mae': 1.49e-04, 'rmse': 7.38e-05, 'compression': 1.00},
    
    # Mixed Fixed-Point
    {'config': 'Fixed2_62_Fixed3_29', 'avg_mae': 6.57e-05, 'max_mae': 1.49e-04, 'rmse': 7.38e-05, 'compression': 1.43},
    {'config': 'Fixed2_46_Fixed3_21', 'avg_mae': 6.57e-05, 'max_mae': 1.49e-04, 'rmse': 7.37e-05, 'compression': 1.90},
    {'config': 'Fixed2_30_Fixed3_13', 'avg_mae': 7.12e-05, 'max_mae': 1.98e-04, 'rmse': 8.27e-05, 'compression': 2.86},
]

# SiLU (Swish) function data
silu_results = [
    # Floating-Point
    {'config': 'FP64_FP64', 'avg_mae': 6.974275e-05, 'max_mae': 1.511148e-04, 'rmse': 8.090011e-05, 'compression': 1.00},
    {'config': 'FP32_FP32', 'avg_mae': 6.97e-05, 'max_mae': 1.51e-04, 'rmse': 8.09e-05, 'compression': 2.00},
    {'config': 'FP16_FP16', 'avg_mae': 8.61e-05, 'max_mae': 2.80e-04, 'rmse': 1.06e-04, 'compression': 4.00},
    {'config': 'FP64_FP32', 'avg_mae': 6.97e-05, 'max_mae': 1.51e-04, 'rmse': 8.09e-05, 'compression': 1.43},
    {'config': 'FP64_FP16', 'avg_mae': 8.61e-05, 'max_mae': 2.80e-04, 'rmse': 1.06e-04, 'compression': 1.82},
    {'config': 'FP32_FP16', 'avg_mae': 8.61e-05, 'max_mae': 2.80e-04, 'rmse': 1.06e-04, 'compression': 2.86},
    
    # Fixed-Point
    {'config': 'Fixed2_14_Fixed2_14', 'avg_mae': 7.39e-05, 'max_mae': 1.74e-04, 'rmse': 8.48e-05, 'compression': 4.00},
    {'config': 'Fixed2_22_Fixed2_22', 'avg_mae': 6.97e-05, 'max_mae': 1.51e-04, 'rmse': 8.09e-05, 'compression': 2.67},
    {'config': 'Fixed2_30_Fixed2_30', 'avg_mae': 6.97e-05, 'max_mae': 1.51e-04, 'rmse': 8.09e-05, 'compression': 2.00},
    {'config': 'Fixed2_38_Fixed2_38', 'avg_mae': 6.97e-05, 'max_mae': 1.51e-04, 'rmse': 8.09e-05, 'compression': 1.60},
    {'config': 'Fixed2_46_Fixed2_46', 'avg_mae': 6.97e-05, 'max_mae': 1.51e-04, 'rmse': 8.09e-05, 'compression': 1.33},
    {'config': 'Fixed2_62_Fixed2_62', 'avg_mae': 6.97e-05, 'max_mae': 1.51e-04, 'rmse': 8.09e-05, 'compression': 1.00},
    
    # Mixed Fixed-Point
    {'config': 'Fixed2_62_Fixed2_30', 'avg_mae': 6.97e-05, 'max_mae': 1.51e-04, 'rmse': 8.09e-05, 'compression': 1.43},
    {'config': 'Fixed2_46_Fixed2_22', 'avg_mae': 6.97e-05, 'max_mae': 1.51e-04, 'rmse': 8.09e-05, 'compression': 1.90},
    {'config': 'Fixed2_30_Fixed2_14', 'avg_mae': 7.39e-05, 'max_mae': 1.74e-04, 'rmse': 8.49e-05, 'compression': 2.86},
]

# Mish function data (uses Fixed2_X_Fixed3_Y, same as GELU)
mish_results = [
    # Floating-Point
    {'config': 'FP64_FP64', 'avg_mae': 6.408668e-05, 'max_mae': 1.481038e-04, 'rmse': 7.267072e-05, 'compression': 1.00},
    {'config': 'FP32_FP32', 'avg_mae': 6.41e-05, 'max_mae': 1.48e-04, 'rmse': 7.27e-05, 'compression': 2.00},
    {'config': 'FP16_FP16', 'avg_mae': 9.92e-05, 'max_mae': 5.74e-04, 'rmse': 1.52e-04, 'compression': 4.00},
    {'config': 'FP64_FP32', 'avg_mae': 6.41e-05, 'max_mae': 1.48e-04, 'rmse': 7.27e-05, 'compression': 1.43},
    {'config': 'FP64_FP16', 'avg_mae': 9.92e-05, 'max_mae': 5.74e-04, 'rmse': 1.51e-04, 'compression': 1.82},
    {'config': 'FP32_FP16', 'avg_mae': 9.92e-05, 'max_mae': 5.74e-04, 'rmse': 1.51e-04, 'compression': 2.86},
    
    # Fixed-Point (Mish uses Fixed2_X_Fixed3_Y, same as GELU)
    {'config': 'Fixed2_14_Fixed3_13', 'avg_mae': 7.25e-05, 'max_mae': 1.99e-04, 'rmse': 8.96e-05, 'compression': 4.00},
    {'config': 'Fixed2_22_Fixed3_21', 'avg_mae': 6.41e-05, 'max_mae': 1.48e-04, 'rmse': 7.27e-05, 'compression': 2.67},
    {'config': 'Fixed2_30_Fixed3_29', 'avg_mae': 6.41e-05, 'max_mae': 1.48e-04, 'rmse': 7.27e-05, 'compression': 2.00},
    {'config': 'Fixed2_38_Fixed3_37', 'avg_mae': 6.41e-05, 'max_mae': 1.48e-04, 'rmse': 7.27e-05, 'compression': 1.60},
    {'config': 'Fixed2_46_Fixed3_45', 'avg_mae': 6.41e-05, 'max_mae': 1.48e-04, 'rmse': 7.27e-05, 'compression': 1.33},
    {'config': 'Fixed2_62_Fixed3_61', 'avg_mae': 6.41e-05, 'max_mae': 1.48e-04, 'rmse': 7.27e-05, 'compression': 1.00},
    
    # Mixed Fixed-Point
    {'config': 'Fixed2_62_Fixed3_29', 'avg_mae': 6.41e-05, 'max_mae': 1.48e-04, 'rmse': 7.27e-05, 'compression': 1.43},
    {'config': 'Fixed2_46_Fixed3_21', 'avg_mae': 6.41e-05, 'max_mae': 1.48e-04, 'rmse': 7.27e-05, 'compression': 1.90},
    {'config': 'Fixed2_30_Fixed3_13', 'avg_mae': 7.25e-05, 'max_mae': 1.99e-04, 'rmse': 8.96e-05, 'compression': 2.86},
]

# HardSwish function data (uses Fixed2_X_Fixed2_Y, same as Tanh and SiLU)
hardswish_results = [
    # Floating-Point
    {'config': 'FP64_FP64', 'avg_mae': 6.930528e-05, 'max_mae': 1.041662e-04, 'rmse': 7.599647e-05, 'compression': 1.00},
    {'config': 'FP32_FP32', 'avg_mae': 6.93e-05, 'max_mae': 1.04e-04, 'rmse': 7.60e-05, 'compression': 2.00},
    {'config': 'FP16_FP16', 'avg_mae': 9.05e-05, 'max_mae': 3.03e-04, 'rmse': 1.14e-04, 'compression': 4.00},
    {'config': 'FP64_FP32', 'avg_mae': 6.93e-05, 'max_mae': 1.04e-04, 'rmse': 7.60e-05, 'compression': 1.43},
    {'config': 'FP64_FP16', 'avg_mae': 9.05e-05, 'max_mae': 3.03e-04, 'rmse': 1.14e-04, 'compression': 1.82},
    {'config': 'FP32_FP16', 'avg_mae': 9.05e-05, 'max_mae': 3.03e-04, 'rmse': 1.14e-04, 'compression': 2.86},
    
    # Fixed-Point (HardSwish uses Fixed2_X_Fixed2_Y, same as Tanh and SiLU)
    {'config': 'Fixed2_14_Fixed2_14', 'avg_mae': 6.88e-05, 'max_mae': 1.51e-04, 'rmse': 7.72e-05, 'compression': 4.00},
    {'config': 'Fixed2_22_Fixed2_22', 'avg_mae': 6.93e-05, 'max_mae': 1.04e-04, 'rmse': 7.60e-05, 'compression': 2.67},
    {'config': 'Fixed2_30_Fixed2_30', 'avg_mae': 6.93e-05, 'max_mae': 1.04e-04, 'rmse': 7.60e-05, 'compression': 2.00},
    {'config': 'Fixed2_38_Fixed2_38', 'avg_mae': 6.93e-05, 'max_mae': 1.04e-04, 'rmse': 7.60e-05, 'compression': 1.60},
    {'config': 'Fixed2_46_Fixed2_46', 'avg_mae': 6.93e-05, 'max_mae': 1.04e-04, 'rmse': 7.60e-05, 'compression': 1.33},
    {'config': 'Fixed2_62_Fixed2_62', 'avg_mae': 6.93e-05, 'max_mae': 1.04e-04, 'rmse': 7.60e-05, 'compression': 1.00},
    
    # Mixed Fixed-Point
    {'config': 'Fixed2_62_Fixed2_30', 'avg_mae': 6.93e-05, 'max_mae': 1.04e-04, 'rmse': 7.60e-05, 'compression': 1.43},
    {'config': 'Fixed2_46_Fixed2_22', 'avg_mae': 6.93e-05, 'max_mae': 1.04e-04, 'rmse': 7.60e-05, 'compression': 1.90},
    {'config': 'Fixed2_30_Fixed2_14', 'avg_mae': 6.88e-05, 'max_mae': 1.51e-04, 'rmse': 7.72e-05, 'compression': 2.86},
]


# Add all data
data_manager.add_function_data('Tanh', tanh_results)
data_manager.add_function_data('Exp', exp_results)
data_manager.add_function_data('GELU', gelu_results)
data_manager.add_function_data('SiLU', silu_results)
data_manager.add_function_data('Mish', mish_results)
data_manager.add_function_data('HardSwish', hardswish_results)

# Show summary
data_manager.get_summary()

# Save data
data_manager.save_data('quantization_data.csv')

print("\n" + "="*80)
print("✓ HardSwish data added successfully!")
print("="*80)
print("\nCurrent dataset includes:")
for func in sorted(data_manager.data['function'].unique()):
    count = len(data_manager.data[data_manager.data['function'] == func])
    print(f"  • {func}: {count} configurations")

print("\n" + "="*80)
print("Fixed-point format summary:")
print("="*80)
print("  • Tanh, SiLU, HardSwish: Fixed2_X_Fixed2_Y (2 int bits)")
print("  • Exp:                    Fixed2_X_Fixed4_Y (4 int bits)")
print("  • GELU, Mish:             Fixed2_X_Fixed3_Y (3 int bits)")
print(f"\nTotal data points: {len(data_manager.data)} (6 functions × 15 configs)")
print("="*80)

✓ Added 15 configurations for 'Tanh'
✓ Added 15 configurations for 'Exp'
✓ Added 15 configurations for 'GELU'
✓ Added 15 configurations for 'SiLU'
✓ Added 15 configurations for 'Mish'
✓ Added 15 configurations for 'HardSwish'

DATA SUMMARY
Total configurations: 90

Functions (6):
  • Exp: 15 configs
  • GELU: 15 configs
  • HardSwish: 15 configs
  • Mish: 15 configs
  • SiLU: 15 configs
  • Tanh: 15 configs

Configuration Types:
  • Fixed: 36 configs
  • Float: 18 configs
  • Mixed: 36 configs

Bit-width Range:
  • Min: 32 bits
  • Max: 128 bits

RMSE Range:
  • Min: 5.17e-05
  • Max: 2.53e-04

✓ Data saved to quantization_data.csv

✓ HardSwish data added successfully!

Current dataset includes:
  • Exp: 15 configurations
  • GELU: 15 configurations
  • HardSwish: 15 configurations
  • Mish: 15 configurations
  • SiLU: 15 configurations
  • Tanh: 15 configurations

Fixed-point format summary:
  • Tanh, SiLU, HardSwish: Fixed2_X_Fixed2_Y (2 int bits)
  • Exp:                    Fixed2_X

In [ ]:
# ============================================================================
# Cell 2: Configuration Analysis (Fixed - Proper Column Initialization)
# ============================================================================

import pandas as pd
import numpy as np

class ConfigurationAnalyzer:
    """
    Analyzes configuration data that has already been parsed by QuantizationDataManager
    No need to re-parse config strings - just use existing fields
    """
    
    def __init__(self, data_manager):
        """
        Parameters:
        -----------
        data_manager : QuantizationDataManager
            Data manager with parsed configuration data
        """
        self.data_manager = data_manager
        self.data = data_manager.data.copy()
        # Add analysis columns immediately
        self._add_analysis_columns()
    
    def _get_alignment_key(self, row):
        """
        Generate alignment key for x-axis positioning
        Configs with same total bits for both endpoint and param share x-position
        
        Examples:
        - FP16_FP16 -> 'FP16_FP16'
        - FP32_FP16 -> 'FP32_FP16'
        - Fixed2_14_Fixed2_14 -> 'Fixed16_Fixed16'
        - Fixed2_14_Fixed4_12 -> 'Fixed16_Fixed16' (same key!)
        - Fixed2_30_Fixed2_14 -> 'Fixed32_Fixed16'
        
        Parameters:
        -----------
        row : Series
            DataFrame row with 'config', 'endpoint_bits', 'param_bits' fields
        
        Returns:
        --------
        str
            Alignment key
        """
        config = row['config']
        
        # Float configs: use original config name
        if 'FP' in config and 'Fixed' not in config:
            return config
        
        # Fixed configs: use total bit-widths
        if 'Fixed' in config and 'FP' not in config:
            return f"Fixed{row['endpoint_bits']}_Fixed{row['param_bits']}"
        
        # Hybrids or unknown: use original
        return config
    
    def _add_analysis_columns(self):
        """
        Add analysis columns to data (called in __init__)
        - alignment_key: For x-axis positioning
        - display_group: For legend grouping
        """
        self.data['alignment_key'] = self.data.apply(self._get_alignment_key, axis=1)
        self.data['display_group'] = self.data['function'] + ': ' + self.data['config_display']
    
    def get_clean_data(self, exclude_hybrids=True):
        """
        Get cleaned data (optionally excluding hybrids)
        
        Parameters:
        -----------
        exclude_hybrids : bool
            Whether to exclude Float-Fixed hybrid configurations
        
        Returns:
        --------
        DataFrame
            Cleaned data
        """
        df = self.data.copy()
        
        if exclude_hybrids:
            df = df[~df['is_hybrid']]
        
        return df
    
    def get_summary_by_type(self, exclude_hybrids=True):
        """
        Get summary statistics grouped by configuration type
        
        Parameters:
        -----------
        exclude_hybrids : bool
            Whether to exclude hybrids
        
        Returns:
        --------
        DataFrame
            Summary statistics
        """
        df = self.get_clean_data(exclude_hybrids)
        
        summary = df.groupby(['config_type']).agg({
            'rmse': ['count', 'mean', 'min', 'max', 'std'],
            'total_bits': ['min', 'max'],
            'function': lambda x: ', '.join(sorted(x.unique()))
        }).round(6)
        
        return summary
    
    def get_summary_by_function(self, exclude_hybrids=True):
        """
        Get summary statistics grouped by function
        
        Parameters:
        -----------
        exclude_hybrids : bool
            Whether to exclude hybrids
        
        Returns:
        --------
        DataFrame
            Summary statistics
        """
        df = self.get_clean_data(exclude_hybrids)
        
        summary = df.groupby(['function']).agg({
            'rmse': ['count', 'mean', 'min', 'max', 'std'],
            'total_bits': ['min', 'max'],
            'config_type': lambda x: ', '.join(sorted(x.unique()))
        }).round(6)
        
        return summary
    
    def get_alignment_groups(self, exclude_hybrids=True):
        """
        Get configurations grouped by alignment key
        Shows which configs will share x-axis position
        
        Parameters:
        -----------
        exclude_hybrids : bool
            Whether to exclude hybrids
        
        Returns:
        --------
        dict
            {alignment_key: [list of (function, config) tuples]}
        """
        df = self.get_clean_data(exclude_hybrids)
        
        groups = {}
        for _, row in df.iterrows():
            key = row['alignment_key']
            if key not in groups:
                groups[key] = []
            groups[key].append((row['function'], row['config']))
        
        return groups
    
    def print_analysis(self, exclude_hybrids=True):
        """
        Print comprehensive analysis of configuration data
        
        Parameters:
        -----------
        exclude_hybrids : bool
            Whether to exclude hybrids in analysis
        """
        df = self.get_clean_data(exclude_hybrids)
        
        print("\n" + "="*80)
        print("CONFIGURATION ANALYSIS")
        print("="*80)
        
        # Basic statistics
        print(f"\nDataset Overview:")
        print(f"  Total data points: {len(df)}")
        print(f"  Functions: {df['function'].nunique()} ({', '.join(sorted(df['function'].unique()))})")
        print(f"  Unique configs: {df['config'].nunique()}")
        print(f"  Unique alignment keys: {df['alignment_key'].nunique()}")
        
        if self.data['is_hybrid'].any():
            hybrid_count = self.data['is_hybrid'].sum()
            print(f"  Hybrids excluded: {hybrid_count}")
        
        # Configuration type distribution
        print(f"\nConfiguration Type Distribution:")
        for ctype in sorted(df['config_type'].unique()):
            count = len(df[df['config_type'] == ctype])
            configs = df[df['config_type'] == ctype]['config_display'].unique()
            print(f"  • {ctype}: {count} data points")
            print(f"    Configs: {', '.join(sorted(set(configs)))}")
        
        # Bit-width range
        print(f"\nBit-Width Range:")
        print(f"  Total bits: {df['total_bits'].min()} - {df['total_bits'].max()}")
        print(f"  Endpoint bits: {df['endpoint_bits'].min()} - {df['endpoint_bits'].max()}")
        print(f"  Parameter bits: {df['param_bits'].min()} - {df['param_bits'].max()}")
        
        # RMSE statistics
        print(f"\nRMSE Statistics:")
        print(f"  Mean: {df['rmse'].mean():.2e}")
        print(f"  Min:  {df['rmse'].min():.2e}")
        print(f"  Max:  {df['rmse'].max():.2e}")
        print(f"  Std:  {df['rmse'].std():.2e}")
        
        # Alignment groups
        print(f"\nAlignment Groups (configs sharing x-position):")
        alignment_groups = self.get_alignment_groups(exclude_hybrids)
        
        for key in sorted(alignment_groups.keys(), key=lambda x: (
            df[df['alignment_key'] == x]['total_bits'].iloc[0],
            x
        )):
            items = alignment_groups[key]
            total_bits = df[df['alignment_key'] == key]['total_bits'].iloc[0]
            print(f"\n  {key} (Total: {total_bits} bits):")
            for func, config in sorted(set(items)):
                display = df[(df['function'] == func) & (df['config'] == config)]['config_display'].iloc[0]
                rmse = df[(df['function'] == func) & (df['config'] == config)]['rmse'].iloc[0]
                print(f"    • {func}: {config} → '{display}' (RMSE: {rmse:.2e})")
        
        print("\n" + "="*80)
    
    def get_plot_data(self, exclude_hybrids=True):
        """
        Prepare data for plotting with proper alignment
        
        Parameters:
        -----------
        exclude_hybrids : bool
            Whether to exclude hybrids
        
        Returns:
        --------
        DataFrame
            Data ready for plotting with alignment_key and display_group
        """
        df = self.get_clean_data(exclude_hybrids)
        
        # Sort by total bits and alignment key for consistent x-axis
        df = df.sort_values(['total_bits', 'alignment_key', 'function'])
        
        return df
    
    def compare_functions(self, config_type=None, exclude_hybrids=True):
        """
        Compare functions side-by-side for same configurations
        
        Parameters:
        -----------
        config_type : str, optional
            Filter by config type ('Float', 'Fixed', 'Mixed')
        exclude_hybrids : bool
            Whether to exclude hybrids
        
        Returns:
        --------
        DataFrame
            Pivot table with functions as columns
        """
        df = self.get_clean_data(exclude_hybrids)
        
        if config_type:
            df = df[df['config_type'] == config_type]
        
        # Pivot to compare functions
        pivot = df.pivot_table(
            index=['alignment_key', 'total_bits'],
            columns='function',
            values='rmse',
            aggfunc='first'
        ).round(8)
        
        return pivot


# ============================================================================
# Test Configuration Analyzer
# ============================================================================

print("\n" + "="*80)
print("Testing Configuration Analyzer (Fixed)")
print("="*80)

# Create analyzer using data manager
analyzer = ConfigurationAnalyzer(data_manager)

# Print comprehensive analysis
analyzer.print_analysis(exclude_hybrids=True)

# Show summary by type
print("\n" + "="*80)
print("Summary by Configuration Type:")
print("="*80)
print(analyzer.get_summary_by_type())

# Show summary by function
print("\n" + "="*80)
print("Summary by Function:")
print("="*80)
print(analyzer.get_summary_by_function())

# Compare functions side-by-side
print("\n" + "="*80)
print("Function Comparison (RMSE) - Fixed Configs:")
print("="*80)
print(analyzer.compare_functions(config_type='Fixed'))

print("\n" + "="*80)
print("Function Comparison (RMSE) - All Configs:")
print("="*80)
comparison = analyzer.compare_functions()
print(comparison)

# Get plot-ready data
plot_data = analyzer.get_plot_data(exclude_hybrids=True)

print("\n" + "="*80)
print("Plot-Ready Data Sample:")
print("="*80)
print(plot_data[['function', 'config', 'config_display', 'alignment_key', 
                 'total_bits', 'rmse']].head(20))

print("\n" + "="*80)
print("✓ Configuration analysis complete!")
print("="*80)
print("\nKey features:")
print("  • Automatic alignment key generation in __init__")
print("  • No redundant config parsing")
print("  • Side-by-side function comparison")
print("  • Plot-ready data with proper sorting")
print("\nAlignment examples:")
print("  • Fixed2_14_Fixed2_14 and Fixed2_14_Fixed4_12 → 'Fixed16_Fixed16' (same x)")
print("  • Fixed2_30_Fixed2_14 and Fixed2_30_Fixed4_12 → 'Fixed32_Fixed16' (same x)")
print("="*80)


PLOT PREPARATION
Total data points: 90
Functions: ['Exp', 'GELU', 'HardSwish', 'Mish', 'SiLU', 'Tanh']
Config types: ['Fixed', 'Float', 'Mixed']
Alignment keys: 15

Plotting:
  Functions: ['Exp', 'GELU', 'HardSwish', 'Mish', 'SiLU', 'Tanh']
  Config types: ['Float', 'Fixed', 'Mixed']
  Alignment keys: 15
  Target RMSE: 1.00e-04 at 40.0% height
  Aligned ratio: 0.8 at 40.0% height

ALIGNMENT VERIFICATION
  Target RMSE position: 0.4000 (target: 0.4000)
  Aligned ratio position: 0.4000 (target: 0.4000)
  Position difference: 0.000000
  ✓ Axes aligned!

✓ Plot saved to: quantization_rmse_analysis.pdf
  Figure size: (26, 13)
  Y-axis scale: Logarithmic
  RMSE range: [5.17e-05, 2.69e-04]
  Ratio range: [0.653, 1.021]
  Target RMSE: 1.00e-04 at 40.0% height
  Aligned ratio: 0.8 at 40.0% height
  Green region: from y_min to target_error


In [144]:
# ============================================================================
# Cell 3: Visualization and Plotting (Optimized Aligned Dual Y-axis)
# ============================================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import LogFormatterSciNotation

# Configure matplotlib for publication-quality figures (ADJUSTED FOR BETTER LAYOUT)
plt.rcParams['font.size'] = 50              
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['axes.linewidth'] = 5.0        
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['axes.labelsize'] = 58         
plt.rcParams['axes.titlesize'] = 62         
plt.rcParams['xtick.labelsize'] = 48        
plt.rcParams['ytick.labelsize'] = 50        
plt.rcParams['legend.fontsize'] = 36        

# Define marker scheme for configuration types
TYPE_MARKERS = {
    'Float': 'o',        # circle - Pure Floating Point
    'Fixed': 's',        # square - Pure Fixed Point
    'Mixed': '^',        # triangle - Mixed configurations
}

# Define color scheme with high contrast and distinguishability
FUNCTION_COLORS = {
    'Tanh': '#E41A1C',      # Bright Red
    # 'Exp': '#377EB8',     # REMOVED - Strong Blue
    'GELU': '#4DAF4A',      # Green
    'SiLU': '#984EA3',      # Purple
    'Mish': '#FF7F00',      # Orange
    'HardSwish': '#A65628', # Brown
    'Sigmoid': '#F781BF',   # Pink
    'Softplus': '#999999',  # Gray
    'ReLU': '#66C2A5',      # Teal
    'SELU': '#8B4513',      # Dark Brown
    'ELU': '#2F4F4F',       # Dark Slate Gray
    'Swish': '#DC143C',     # Crimson
}

# Define display name mapping
FUNCTION_DISPLAY_NAMES = {
    'HardSwish': 'HSwish',
    'GELU': 'GELU',
    # 'Exp': 'Exp',         # REMOVED
    'Mish': 'Mish',
    'Tanh': 'Tanh',
    'SiLU': 'SiLU',
    'Sigmoid': 'Sigmoid',
    'Softplus': 'Softplus',
    'ReLU': 'ReLU',
    'SELU': 'SELU',
    'ELU': 'ELU',
    'Swish': 'Swish',
}


class QuantizationVisualizer:
    """
    Visualization class for quantization analysis
    Works with updated ConfigurationAnalyzer
    """
    
    def __init__(self, analyzer):
        """
        Parameters:
        -----------
        analyzer : ConfigurationAnalyzer
            Analyzer with parsed and analyzed data
        """
        self.analyzer = analyzer
    
    def plot_pareto_optimal_rmse(self, 
                                 functions=None,
                                 config_types=None,
                                 exclude_hybrids=True,
                                 exclude_configs=None,
                                 output_file='pareto_optimal_rmse.pdf',
                                 figsize=(28, 13),        
                                 use_log_scale=True,
                                 target_error=1e-4,
                                 show_ratio=True,
                                 aligned_ratio=0.8,
                                 target_position=0.40):
        """
        Create Pareto-optimal plot with RMSE as primary metric
        
        Parameters:
        -----------
        functions : list of str, optional
            Functions to include (default: all)
        config_types : list of str, optional
            Config types to include (default: ['Float', 'Fixed', 'Mixed'])
        exclude_hybrids : bool
            Exclude Float-Fixed hybrid configurations
        exclude_configs : list of str, optional
            List of config_display names to exclude
        output_file : str
            Output PDF filename
        figsize : tuple
            Figure size (width, height)
        use_log_scale : bool
            Use log scale for y-axis
        target_error : float
            Target RMSE threshold
        show_ratio : bool
            Show MAE/RMSE ratio on secondary y-axis
        aligned_ratio : float
            MAE/RMSE ratio value that aligns with target_error (default: 0.8)
        target_position : float
            Relative position (0-1) where target_error appears (default: 0.40 for 40% height)
        """
        
        # Get clean data
        df = self.analyzer.get_clean_data(exclude_hybrids=exclude_hybrids)
        
        # REMOVE Exp function
        df = df[df['function'] != 'Exp']
        
        # Filter by functions
        if functions:
            df = df[df['function'].isin(functions)]
        
        # Filter by config types
        if config_types:
            df = df[df['config_type'].isin(config_types)]
        
        # Exclude specific configurations
        if exclude_configs:
            df = df[~df['config_display'].isin(exclude_configs)]
            print(f"\nExcluding configurations: {exclude_configs}")
        
        if df.empty:
            print("✗ No data to plot!")
            return
        
        print(f"\n{'='*80}")
        print("PLOT PREPARATION")
        print(f"{'='*80}")
        print(f"Total data points: {len(df)}")
        print(f"Functions: {sorted(df['function'].unique())}")
        print(f"Config types: {sorted(df['config_type'].unique())}")
        print(f"Alignment keys: {df['alignment_key'].nunique()}")
        
        # Get unique alignment keys in sorted order
        type_order = {'Float': 0, 'Fixed': 1, 'Mixed': 2}
        df['type_order'] = df['config_type'].map(type_order)
        
        # Sort by type and total bits (descending for bits)
        alignment_df = df.drop_duplicates('alignment_key').sort_values(
            ['type_order', 'total_bits'],
            ascending=[True, False]
        )
        
        alignment_keys = alignment_df['alignment_key'].tolist()
        config_types_ordered = alignment_df['config_type'].tolist()
        
        # Create x-position mapping
        x_positions = {key: i for i, key in enumerate(alignment_keys)}
        
        # Create display labels (use config_display) - REMOVE "Mixed" PREFIX
        display_map = df.drop_duplicates('alignment_key').set_index('alignment_key')['config_display'].to_dict()
        display_labels = []
        for key in alignment_keys:
            label = display_map.get(key, key)
            # Remove "Mixed" prefix if it exists
            if label.startswith('Mixed'):
                label = label.replace('Mixed', '', 1).strip()
            display_labels.append(label)
        
        # Create figure with adjusted layout for larger main plot area
        fig = plt.figure(figsize=figsize)
        ax1 = fig.add_axes([0.08, 0.12, 0.72, 0.83])
        
        # Get unique values for plotting
        plot_functions = sorted(df['function'].unique())
        plot_types = sorted(df['config_type'].unique(), key=lambda x: type_order.get(x, 99))
        
        # Auto-assign colors if function not in predefined colors
        FALLBACK_COLORS = ['#8B0000', '#00008B', '#006400', '#8B008B', '#FF4500', 
                          '#4B0082', '#2F4F4F', '#8B4513', '#483D8B', '#2E8B57']
        for i, func in enumerate(plot_functions):
            if func not in FUNCTION_COLORS:
                FUNCTION_COLORS[func] = FALLBACK_COLORS[i % len(FALLBACK_COLORS)]
        
        print(f"\nPlotting:")
        print(f"  Functions: {plot_functions}")
        print(f"  Config types: {plot_types}")
        print(f"  Alignment keys: {len(alignment_keys)}")
        print(f"  Target RMSE: {target_error:.2e} at {target_position:.1%} height")
        print(f"  Aligned ratio: {aligned_ratio} at {target_position:.1%} height")
        
        # ====================================================================
        # CALCULATE Y-AXIS RANGES FOR PERFECT ALIGNMENT
        # ====================================================================
        
        # Primary axis (RMSE) range
        rmse_min = df['rmse'].min()
        rmse_max = df['rmse'].max()
        
        if use_log_scale:
            log_target = np.log10(target_error)
            log_min_data = np.log10(rmse_min)
            log_max_data = np.log10(rmse_max)
            
            log_range_estimate = (log_target - log_min_data) / target_position
            
            log_y_min = log_target - target_position * log_range_estimate
            log_y_max = log_target + (1 - target_position) * log_range_estimate
            
            y_min = 10 ** log_y_min
            y_max = 10 ** log_y_max
            
            if rmse_min < y_min:
                y_min = rmse_min * 0.5
                log_y_min = np.log10(y_min)
                log_range_new = (log_target - log_y_min) / target_position
                log_y_max = log_y_min + log_range_new
                y_max = 10 ** log_y_max
                
            if rmse_max > y_max:
                y_max = rmse_max * 2.0
                log_y_max = np.log10(y_max)
                log_range_new = (log_y_max - log_target) / (1 - target_position)
                log_y_min = log_y_max - log_range_new
                y_min = 10 ** log_y_min
                
        else:
            y_min = 0
            y_max = target_error / target_position
            
            if rmse_max > y_max:
                y_max = rmse_max * 1.2
                y_min = 0
        
        # Secondary axis (MAE/RMSE ratio) range
        if show_ratio and 'mae_rmse_ratio' in df.columns:
            ratio_min = df['mae_rmse_ratio'].min()
            ratio_max = df['mae_rmse_ratio'].max()
            
            ratio_range_estimate = (aligned_ratio - ratio_min) / target_position
            
            ratio_y_min = aligned_ratio - target_position * ratio_range_estimate
            ratio_y_max = aligned_ratio + (1 - target_position) * ratio_range_estimate
            
            if ratio_y_min < 0:
                ratio_y_min = 0
                ratio_y_max = aligned_ratio / target_position
            
            if ratio_min < ratio_y_min:
                ratio_y_min = max(0, ratio_min * 0.95)
                ratio_range_new = (aligned_ratio - ratio_y_min) / target_position
                ratio_y_max = ratio_y_min + ratio_range_new
                
            if ratio_max > ratio_y_max:
                ratio_y_max = ratio_max * 1.05
                ratio_range_new = (ratio_y_max - aligned_ratio) / (1 - target_position)
                ratio_y_min = ratio_y_max - ratio_range_new
                if ratio_y_min < 0:
                    ratio_y_min = 0
                    ratio_y_max = aligned_ratio / target_position
        
        # ====================================================================
        # PLOT ACCEPTABLE REGION
        # ====================================================================
        
        x_min = -0.5
        x_max = len(alignment_keys) - 0.5
        
        acceptable_region = ax1.axhspan(y_min, target_error, 
                                        xmin=0,
                                        xmax=1,
                                        color='green', 
                                        alpha=0.15, 
                                        label='Acceptable Region',
                                        zorder=1)
        
        # ====================================================================
        # PLOT DATA POINTS (FURTHER ENLARGED)
        # ====================================================================
        
        for func in plot_functions:
            for cfg_type in plot_types:
                subset = df[(df['function'] == func) & 
                           (df['config_type'] == cfg_type)]
                
                if subset.empty:
                    continue
                
                x_vals = [x_positions[key] for key in subset['alignment_key']]
                y_vals = subset['rmse'].values
                
                ax1.scatter(x_vals, y_vals,
                           marker=TYPE_MARKERS.get(cfg_type, 'o'),
                           color=FUNCTION_COLORS[func],
                           s=800,                    
                           alpha=0.8,
                           edgecolors='black',
                           linewidth=4.0,            
                           label=f'{func}_{cfg_type}',
                           zorder=3)
        
        # Add target RMSE line
        target_line = ax1.axhline(y=target_error, 
                                  color='red', 
                                  linestyle='--', 
                                  linewidth=6.0,         
                                  alpha=0.8,
                                  label=f'Target RMSE = {target_error:.0e}',
                                  zorder=2)
        
        # ====================================================================
        # CONFIGURE PRIMARY AXIS
        # ====================================================================
        
        ax1.set_ylim(y_min, y_max)

        if use_log_scale:
            ax1.set_yscale('log')
            ax1.yaxis.set_major_formatter(LogFormatterSciNotation())
            ax1.set_ylabel('RMSE [log]', 
                        fontweight='bold', fontsize=58, labelpad=25)
        else:
            ax1.set_ylabel('Root Mean Square Error (RMSE)', 
                        fontweight='bold', fontsize=58, labelpad=25)

        ax1.set_xlim(x_min, x_max)
        ax1.set_xticks(range(len(alignment_keys)))
        ax1.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=48)
        ax1.set_xlabel('Configuration: Float → Fixed → Mixed',
                    fontweight='bold', fontsize=58, labelpad=20)

        ax1.grid(True, alpha=0.3, linestyle='--', linewidth=3.0)
        ax1.tick_params(axis='both', which='major', labelsize=50, width=5.0, length=16)
        
        # ====================================================================
        # ADD SECONDARY Y-AXIS
        # ====================================================================
        
        if show_ratio and 'mae_rmse_ratio' in df.columns:
            ax2 = ax1.twinx()
            
            ax2.set_ylim(ratio_y_min, ratio_y_max)
            ax2.set_ylabel('EU Ratio (MAE/RMSE)', 
                          fontweight='bold', fontsize=58, rotation=270, labelpad=80)
            ax2.tick_params(axis='y', labelsize=50, width=5.0, length=16)
            
            # Plot ratio data points (ENLARGED X markers)
            for func in plot_functions:
                for cfg_type in plot_types:
                    subset = df[(df['function'] == func) & 
                               (df['config_type'] == cfg_type)]
                    
                    if subset.empty:
                        continue
                    
                    x_vals = [x_positions[key] for key in subset['alignment_key']]
                    ratio_vals = subset['mae_rmse_ratio'].values
                    
                    ax2.scatter(x_vals, ratio_vals,
                               marker='x',
                               color=FUNCTION_COLORS[func],
                               s=650,                    
                               alpha=0.6,
                               linewidth=6.5,            
                               zorder=3)
            
            perfect_line = ax2.axhline(y=1.0,
                                       color='blue',
                                       linestyle='--',
                                       linewidth=6.0,            
                                       alpha=0.7,
                                       label='Perfect Uniformity',
                                       zorder=2)
        
        # ====================================================================
        # ADD VERTICAL SEPARATORS
        # ====================================================================
        
        prev_type = None
        for i, (key, cfg_type) in enumerate(zip(alignment_keys, config_types_ordered)):
            if prev_type is not None and cfg_type != prev_type:
                ax1.axvline(x=i-0.5, color='gray', linestyle='-', 
                           linewidth=5.5, alpha=0.5, zorder=2)
            prev_type = cfg_type
        
        # ====================================================================
        # CREATE LEGENDS - FUNCTIONS IN 1 COLUMN, CONFIG TYPES SEPARATE
        # ====================================================================
        
        from matplotlib.lines import Line2D
        from matplotlib.patches import Patch
        
        # Legend 1: Functions only (1 column)
        function_handles = []
        function_labels = []
        
        for func in plot_functions:
            display_name = FUNCTION_DISPLAY_NAMES.get(func, func)
            function_handles.append(Line2D([0], [0], 
                                          marker='o',              
                                          color='w',               
                                          markerfacecolor=FUNCTION_COLORS[func],  
                                          markeredgecolor='black', 
                                          markersize=24,           
                                          linewidth=0,             
                                          markeredgewidth=2.5))
            function_labels.append(display_name)
        
        legend1 = ax1.legend(function_handles, function_labels,
                            loc='upper left',
                            bbox_to_anchor=(1.12, 1.05),
                            ncol=1,                         # 2 → 1 (改为 1 列)
                            fontsize=36,                    
                            frameon=True,
                            framealpha=0.9,
                            edgecolor='black',
                            fancybox=False,
                            shadow=False,
                            title='Functions',              # 修改标题
                            title_fontsize=40,              
                            handlelength=2.0,               
                            handleheight=1.5,               
                            labelspacing=0.3,               
                            handletextpad=0.5)
        legend1.get_frame().set_linewidth(3.5)
        ax1.add_artist(legend1)
        
        # Legend 2: Config Types + EU Ratio
        config_handles = []
        config_labels = []
        
        for cfg_type in plot_types:
            config_handles.append(Line2D([0], [0], marker=TYPE_MARKERS[cfg_type],
                                      color='w', markerfacecolor='gray',
                                      markeredgecolor='black',
                                      markersize=24, linewidth=0,
                                      markeredgewidth=2.5))
            config_labels.append(cfg_type)
        
        # Add X marker explanation
        config_handles.append(Line2D([0], [0], marker='x',
                                  color='gray',
                                  markerfacecolor='gray',
                                  markeredgecolor='gray',
                                  markersize=24, linewidth=5.0,
                                  markeredgewidth=5.0))
        config_labels.append('EU Ratio')
        
        # Calculate position based on number of functions
        # Approximate height per function item: 0.06
        functions_height = len(plot_functions) * 0.06 + 0.15  # extra for title and margins
        legend2_y_position = 1.05 - functions_height - 0.05   # small gap
        
        legend2 = ax1.legend(config_handles, config_labels,
                            loc='upper left',
                            bbox_to_anchor=(1.12, legend2_y_position),
                            ncol=1,
                            fontsize=36,
                            frameon=True,
                            framealpha=0.9,
                            edgecolor='black',
                            fancybox=False,
                            shadow=False,
                            title='Config Types',           # 修改标题
                            title_fontsize=40,
                            handlelength=2.0,
                            handleheight=1.5,
                            labelspacing=0.3,
                            handletextpad=0.5)
        legend2.get_frame().set_linewidth(3.5)
        ax1.add_artist(legend2)
        
        # Legend 3: Thresholds
        threshold_handles = []
        threshold_labels = []
        
        threshold_handles.append(Patch(facecolor='green', alpha=0.15,
                                      edgecolor='black', linewidth=2.5))
        threshold_labels.append('Acceptable Region')
        
        threshold_handles.append(Line2D([0], [0], color='red', linestyle='--', 
                                       linewidth=5.0, alpha=0.8))
        threshold_labels.append(f'Target AvgMAE')
        
        if show_ratio:
            threshold_handles.append(Line2D([0], [0], color='blue', linestyle='--', 
                                           linewidth=5.0, alpha=0.7))
            threshold_labels.append('Perfect Uniformity')
        
        # Calculate position for thresholds legend
        config_types_height = (len(plot_types) + 1) * 0.06 + 0.15  # +1 for EU Ratio
        legend3_y_position = legend2_y_position - config_types_height - 0.05
        
        legend3 = ax1.legend(threshold_handles, threshold_labels,
                            loc='upper left',
                            bbox_to_anchor=(1.01, -0.05),
                            ncol=1,
                            fontsize=36,
                            frameon=True,
                            framealpha=0.9,
                            edgecolor='black',
                            fancybox=False,
                            shadow=False,
                            title='Thresholds',
                            title_fontsize=40,
                            handlelength=2.0,
                            handleheight=1.5,
                            labelspacing=0.3,
                            handletextpad=0.5)
        legend3.get_frame().set_linewidth(3.5)
        
        # ====================================================================
        # VERIFICATION
        # ====================================================================
        
        if show_ratio:
            if use_log_scale:
                actual_rmse_position = (np.log10(target_error) - np.log10(y_min)) / (np.log10(y_max) - np.log10(y_min))
            else:
                actual_rmse_position = (target_error - y_min) / (y_max - y_min)
            
            actual_ratio_position = (aligned_ratio - ratio_y_min) / (ratio_y_max - ratio_y_min)
            
            print(f"\n{'='*80}")
            print("ALIGNMENT VERIFICATION")
            print(f"{'='*80}")
            print(f"  Target RMSE position: {actual_rmse_position:.4f} (target: {target_position:.4f})")
            print(f"  Aligned ratio position: {actual_ratio_position:.4f} (target: {target_position:.4f})")
            print(f"  Position difference: {abs(actual_rmse_position - actual_ratio_position):.6f}")
            print(f"  ✓ Axes aligned!" if abs(actual_rmse_position - actual_ratio_position) < 0.01 else "  ✗ Alignment issue!")
        
        # ====================================================================
        # SET TITLE AND FINALIZE
        # ====================================================================
        
        title_parts = ["RMSE and Error Uniformity Analysis"]
        
        plt.title(' '.join(title_parts), 
                 fontweight='bold', fontsize=62, pad=35)

        plt.savefig(output_file, 
                   format='pdf', 
                   dpi=300, 
                   bbox_inches='tight',
                   pad_inches=0.3)
        
        print(f"\n✓ Plot saved to: {output_file}")
        print(f"  Figure size: {figsize}")
        print(f"  Y-axis scale: {'Logarithmic' if use_log_scale else 'Linear'}")
        print(f"  RMSE range: [{y_min:.2e}, {y_max:.2e}]")
        if show_ratio:
            print(f"  Ratio range: [{ratio_y_min:.3f}, {ratio_y_max:.3f}]")
        print(f"  Target RMSE: {target_error:.2e} at {target_position:.1%} height")
        if show_ratio:
            print(f"  Aligned ratio: {aligned_ratio} at {target_position:.1%} height")
        print(f"  Green region: from y_min to target_error")
        
        plt.close()


# ============================================================================
# Example Usage
# ============================================================================

if __name__ == "__main__":
    viz = QuantizationVisualizer(analyzer)
    
    viz.plot_pareto_optimal_rmse(
        exclude_hybrids=True,
        output_file='quantization_rmse_analysis.pdf',
        figsize=(28, 13),
        use_log_scale=True,
        target_error=1e-4,
        show_ratio=True,
        aligned_ratio=0.8,
        target_position=0.40
    )


PLOT PREPARATION
Total data points: 75
Functions: ['GELU', 'HardSwish', 'Mish', 'SiLU', 'Tanh']
Config types: ['Fixed', 'Float', 'Mixed']
Alignment keys: 15

Plotting:
  Functions: ['GELU', 'HardSwish', 'Mish', 'SiLU', 'Tanh']
  Config types: ['Float', 'Fixed', 'Mixed']
  Alignment keys: 15
  Target RMSE: 1.00e-04 at 40.0% height
  Aligned ratio: 0.8 at 40.0% height

ALIGNMENT VERIFICATION
  Target RMSE position: 0.4000 (target: 0.4000)
  Aligned ratio position: 0.4000 (target: 0.4000)
  Position difference: 0.000000
  ✓ Axes aligned!

✓ Plot saved to: quantization_rmse_analysis.pdf
  Figure size: (28, 13)
  Y-axis scale: Logarithmic
  RMSE range: [4.42e-05, 3.40e-04]
  Ratio range: [0.653, 1.021]
  Target RMSE: 1.00e-04 at 40.0% height
  Aligned ratio: 0.8 at 40.0% height
  Green region: from y_min to target_error
